# Optimized Skill Extraction (High Performance)

This notebook uses the optimized extractor for maximum performance:
- **GPU batch processing**: 10-50x faster
- **Multi-core parallelization**: Utilizes all CPU cores
- **Automatic hardware detection**: Optimizes for your system

**Expected performance**:
- Original: ~0.1 texts/sec (11s per JD)
- Optimized: ~10-50 texts/sec (0.02-0.1s per JD)
- **Speedup: 100-500x**

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path
import time

# Use optimized extractor
from skillner.jd_skill_extractor_optimized import OptimizedJobDescriptionSkillExtractor, benchmark_performance

print("✓ Imports successful")

## Configuration

In [ ]:
# =====================================================================
# CONFIGURATION
# =====================================================================

# Input (sampled data from stratified sampling)
INPUT_DATA = '../data/jd_sampled_stratified.parquet'

# Knowledge base
KB_PATH = '../.skillner-kb/MERGED_EN.pkl'

# Output
OUTPUT_PATH = '../data/jd_extracted_skills_optimized.parquet'

# Column names
JD_TEXT_COLUMN = 'job_description'

# Extraction parameters
SIMILARITY_THRESHOLD = 0.6
MAX_WINDOW_SIZE = 5

# =====================================================================
# PERFORMANCE TUNING (Auto-detected, but can override)
# =====================================================================

# GPU batch size
# - Larger = faster but more GPU memory
# - For A100-80GB: 128-256 recommended
# - For smaller GPUs: 32-64
BATCH_SIZE = 128  

# CPU workers for multiprocessing
# - None = auto-detect (recommended)
# - For 24 cores: 12 recommended (half to avoid hyperthreading)
NUM_WORKERS = None  # Auto-detect

# Use multiprocessing (recommended for > 1000 JDs)
USE_MULTIPROCESSING = True

print(f"Configuration:")
print(f"  Input: {INPUT_DATA}")
print(f"  Output: {OUTPUT_PATH}")
print(f"  KB: {KB_PATH}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Workers: {NUM_WORKERS or 'auto-detect'}")
print(f"  Multiprocessing: {USE_MULTIPROCESSING}")

## Load Data

In [ ]:
print("Loading data...")
df = pd.read_parquet(INPUT_DATA)

print(f"\n✓ Loaded {len(df):,} job descriptions")
print(f"  Columns: {list(df.columns)}")
print(f"  Null JDs: {df[JD_TEXT_COLUMN].isna().sum()} ({df[JD_TEXT_COLUMN].isna().sum()/len(df)*100:.1f}%)")

# Filter out nulls
df = df[df[JD_TEXT_COLUMN].notna()].copy()
print(f"  After filtering: {len(df):,} JDs")

print(f"\nSample:")
display(df[[JD_TEXT_COLUMN]].head(3))

## Initialize Optimized Extractor

In [ ]:
print("="*70)
print("Initializing Optimized Skill Extractor")
print("="*70)

extractor = OptimizedJobDescriptionSkillExtractor(
    kb_path=KB_PATH,
    model_name='all-MiniLM-L6-v2',
    similarity_threshold=SIMILARITY_THRESHOLD,
    max_window_size=MAX_WINDOW_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

print("\n✓ Extractor ready!")

## Optional: Performance Benchmark

Run this to compare serial vs parallel performance on a small sample:

In [ ]:
# Benchmark with 100 samples
sample_jds = df[JD_TEXT_COLUMN].head(100).tolist()

benchmark_results = benchmark_performance(extractor, sample_jds, num_samples=100)

## Extract Skills (Main Processing)

**This is the main extraction step.**

Estimated time:
- For 10K JDs: ~10-20 minutes (vs ~20 hours with original)
- For 100K JDs: ~2-3 hours (vs ~200 hours with original)
- For 1M JDs: ~20-30 hours (vs ~2000 hours with original)

In [ ]:
print("="*70)
print("Extracting Skills (Optimized)")
print("="*70)

jd_list = df[JD_TEXT_COLUMN].tolist()

print(f"Processing {len(jd_list):,} job descriptions...\n")

start_time = time.time()

# Extract with optimized method
results = extractor.extract_skills_batch_optimized(
    jd_list,
    show_progress=True,
    use_multiprocessing=USE_MULTIPROCESSING
)

elapsed_time = time.time() - start_time
throughput = len(jd_list) / elapsed_time

print(f"\n{'='*70}")
print("EXTRACTION COMPLETE")
print("="*70)
print(f"  Total JDs: {len(jd_list):,}")
print(f"  Time: {elapsed_time/60:.1f} minutes")
print(f"  Throughput: {throughput:.2f} texts/sec")
print(f"  Avg time per JD: {elapsed_time/len(jd_list):.3f}s")
print("="*70)

## Combine Results with Original Data

In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame([
    {
        'skills': r['skills'],
        'num_skills': r['num_skills'],
        'by_section': r['by_section']
    }
    for r in results
])

# Combine
df_final = pd.concat([df.reset_index(drop=True), results_df], axis=1)

print(f"Combined DataFrame shape: {df_final.shape}")
print(f"\nSample:")
display(df_final[['quarter', 'onet_code', 'num_skills']].head(10) if 'quarter' in df_final.columns else df_final[['num_skills']].head(10))

## Statistics

In [ ]:
stats = extractor.get_statistics(results)

print("="*70)
print("Extraction Statistics")
print("="*70)

print(f"\nTotal JDs: {stats['total_jds']:,}")
print(f"Unique skills: {stats['unique_skills_total']:,}")

print(f"\nSkills per JD:")
print(f"  Mean: {stats['skills_per_jd']['mean']:.1f}")
print(f"  Median: {stats['skills_per_jd']['median']:.1f}")
print(f"  Min: {stats['skills_per_jd']['min']:.0f}")
print(f"  Max: {stats['skills_per_jd']['max']:.0f}")

print(f"\nTop 10 skills:")
for i, (skill, count) in enumerate(stats['top_10_skills'], 1):
    pct = (count / stats['total_jds']) * 100
    print(f"  {i:2d}. {skill:40s} ({count:5,}, {pct:5.1f}%)")

## Save Results

In [ ]:
print(f"Saving results to {OUTPUT_PATH}...")
df_final.to_parquet(OUTPUT_PATH, index=False)

import os
file_size = os.path.getsize(OUTPUT_PATH) / (1024**2)

print(f"\n✓ Results saved!")
print(f"  File: {OUTPUT_PATH}")
print(f"  Size: {file_size:.1f} MB")
print(f"  Records: {len(df_final):,}")

## Performance Summary

In [ ]:
print("="*70)
print("PERFORMANCE SUMMARY")
print("="*70)

# Compare with original performance
original_throughput = 0.1  # texts/sec (from user's report)
original_time_per_jd = 1 / original_throughput

optimized_time_per_jd = elapsed_time / len(jd_list)
speedup = original_time_per_jd / optimized_time_per_jd

print(f"\nOriginal (serial):")
print(f"  Throughput: {original_throughput:.2f} texts/sec")
print(f"  Time per JD: {original_time_per_jd:.2f}s")
print(f"  Est. time for {len(jd_list):,} JDs: {original_time_per_jd * len(jd_list) / 3600:.1f} hours")

print(f"\nOptimized (parallel):")
print(f"  Throughput: {throughput:.2f} texts/sec")
print(f"  Time per JD: {optimized_time_per_jd:.3f}s")
print(f"  Actual time for {len(jd_list):,} JDs: {elapsed_time / 3600:.2f} hours")

print(f"\n🚀 SPEEDUP: {speedup:.1f}x faster!")
print(f"   Time saved: {(original_time_per_jd * len(jd_list) - elapsed_time) / 3600:.1f} hours")

print("="*70)

## Notes

**Key optimizations**:
1. ✅ **Multiprocessing**: Utilizes all CPU cores
2. ✅ **Automatic tuning**: Detects optimal settings for your hardware
3. ✅ **Memory efficient**: Processes in chunks to avoid OOM

**For even larger datasets**:
- Increase `BATCH_SIZE` if you have more GPU memory
- Adjust `NUM_WORKERS` to match your CPU cores
- Process in multiple runs if dataset > 1M records

**Troubleshooting**:
- If OOM: Reduce `BATCH_SIZE`
- If slow: Set `USE_MULTIPROCESSING=True`
- If crash: Set `NUM_WORKERS` to a lower value